# DataMate LLM on Colab (stand-in for the Modal endpoint)

Same model, served name and vLLM flags as `deploy/modal_vllm.py`, so the local LLM cache (keyed by model name) stays valid.
Only the model runs here; data and the pipeline stay on the local machine, which calls this endpoint through `LLM_BASE_URL` / `LLM_API_KEY` in `.env`.

1. Runtime -> Change runtime type -> **G4 GPU** (RTX PRO 6000 Blackwell, fastest, same architecture as the 5090) or L4 (cheaper, slower). A100 also works; T4 does not.
2. Add a Colab secret (key icon on the left) named `VLLM_API_KEY` with the same long random string as `LLM_API_KEY` in your local `.env`. Never paste it into a cell.
3. Run all cells. The last-but-one prints the public URL to put in `.env`.
4. When done: run the stop cell, then **Runtime -> Disconnect and delete runtime** (compute units are billed while connected).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Same vLLM version as the Modal image and the local docker compose (~5 min).
!pip install -q vllm==0.32.0 "huggingface_hub[hf_transfer]"

In [ ]:
import os
import subprocess

from google.colab import userdata

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
gpu_mb = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]
).split()[0])
if gpu_mb >= 40000:
    # G4 (RTX PRO 6000 Blackwell, 96 GB) or A100: same settings as Modal; eval --workers 24.
    MAX_LEN, EXTRA = "16384", ["--gpu-memory-utilization", "0.92"]
else:
    # L4 (24 GB): ~15 GB of FP8 weights leave little KV cache. 12k still fits a thinking
    # parse (~6k prompt + 3k reasoning); 16 sequences keep CUDA graphs small. --workers 8.
    MAX_LEN, EXTRA = "12288", ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "16"]
print("GPU MB", gpu_mb, "max len", MAX_LEN)
cmd = [
    "vllm", "serve", "Qwen/Qwen3-14B-FP8",
    "--served-model-name", "qwen3-14b",
    "--max-model-len", MAX_LEN,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api-key", userdata.get("VLLM_API_KEY"),
    "--enable-prefix-caching",
    "--reasoning-parser", "qwen3",
    *EXTRA,
]
vllm_log = open("/content/vllm.log", "w")
vllm = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
print("vLLM starting, pid", vllm.pid)

In [ ]:
# Wait for the model download and load (first time ~5-10 min).
import time
import urllib.request

for _ in range(180):
    if vllm.poll() is not None:
        print(open("/content/vllm.log").read()[-4000:])
        raise RuntimeError("vLLM exited, log above")
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        time.sleep(10)
!tail -n 5 /content/vllm.log

In [ ]:
# Public HTTPS URL through a Cloudflare quick tunnel (no account needed).
# The API key above protects the endpoint; the URL changes every session.
import re

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(30):
    time.sleep(2)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found[0]
        break
print("Put this in .env:")
print(f"LLM_BASE_URL={url}/v1")

In [ ]:
# Stop serving (then Runtime -> Disconnect and delete runtime).
tunnel.terminate()
vllm.terminate()
print("stopped")